In [1]:
!pip install pdfplumber pandas tqdm

  Using cached pdfplumber-0.11.8-py3-none-any.whl.metadata (43 kB)
  Using cached pdfminer_six-20251107-py3-none-any.whl.metadata (4.2 kB)
Using cached pdfplumber-0.11.8-py3-none-any.whl (60 kB)
Using cached pdfminer_six-20251107-py3-none-any.whl (5.6 MB)
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 3.5/3.5 MB 20.8 MB/s  0:00:00
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   -------------------------------------- - 6.8/7.0 MB 42.0 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 33.2 MB/s  0:00:00
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ---------------------------------------- 3.1/3.1 MB 22.7 MB/s  0:00:00

   ---------------------------------------- 0/6 [tqdm]
   ------ --------------------------------- 1/6 [pypdfium2]
   ------ --------------------------------- 1/6 [pypdfium2]
   ------ --------------------------------- 1/6 [pypdfium2]
  

In [34]:
# 2번 셀

import re
import os
from pathlib import Path

import pdfplumber
import pandas as pd
from tqdm import tqdm

# "제1장 총칙" 같은 장(章) 패턴
CHAPTER_PATTERN = re.compile(r'^제\s*\d+\s*장\s*.*')

# "제1조(목적)" / "제35조 과징금 처분 등" 같은 조(條) 패턴
ARTICLE_PATTERN = re.compile(
    r'^제\s*\d+\s*조(?:\s*\(.*?\))?\s*$'
)
# ①, ②, ③, ... 항 번호
CIRCLED_DIGITS = "①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳"
CLAUSE_PATTERN = re.compile(rf'^[{CIRCLED_DIGITS}]\s*')

# "제35조(과징금 처분 등) ① 국토교통부장관 ..." 전체 라인 처리용
ARTICLE_TITLE_PATTERN = re.compile(
    r'^제\s*(\d+)\s*조(?:\((.+?)\))?\s*(.*)$'
)


In [35]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """pdfplumber로 PDF 전체 텍스트 추출"""
    text_chunks = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text_chunks.append(page.extract_text() or "")
    return "\n".join(text_chunks)


def normalize_lines(text: str, law_name: str = None):
    """
    줄 단위로 쪼개고, 전각 공백 제거 + 양쪽 공백 제거.
    페이지 헤더/푸터 텍스트는 필터링.
    """
    lines = text.splitlines()
    norm_lines = []
    for line in lines:
        # 전각 공백 제거 & 트림
        line = line.replace('\u3000', ' ')
        line = line.strip()
        if not line:
            continue

        # ----- 헤더/푸터 필터링 -----
        # 법령명 단독 줄 (페이지 하단에 반복)
        if law_name and line == law_name:
            continue
        # 법제처 / 국가법령정보센터
        if "국가법령정보센터" in line or "법제처" in line:
            continue
        # 페이지 번호만 있는 줄 (1, 2, 3 ...)
        if re.fullmatch(r'\d+', line):
            continue
        # ---------------------------

        norm_lines.append(line)

    return norm_lines

In [36]:
def split_clause_start(line: str):
    """
    줄 맨 앞에 ①, ②, ... 같은 항 번호가 있으면
    (항번호, 나머지본문) 으로 쪼개서 반환.
    없으면 (None, 원래줄) 반환.
    """
    line = line.lstrip()
    m = CLAUSE_PATTERN.match(line)
    if not m:
        return None, line
    clause_char = m.group(0).strip()[0]  # ① 같은 한 글자
    rest = CLAUSE_PATTERN.sub("", line).strip()
    return clause_char, rest


def parse_article_title(line: str):
    """
    '제35조(과징금 처분 등) ① 국토교통부장관...' 형식 처리:
    - article_number = '제35조'
    - article_title  = '과징금 처분 등'
    - body_start     = '① 국토교통부장관...'  (같은 줄에 붙은 본문)
    
    '제1조(목적) 이 법은 ...' 도:
    - article_number = '제1조'
    - article_title  = '목적'
    - body_start     = '이 법은 ...'
    """
    m = ARTICLE_TITLE_PATTERN.match(line)
    if not m:
        # 패턴 안 맞으면 전체를 제목 취급
        return line.strip(), None, None

    num, title_in_paren, rest = m.groups()
    article_number = f"제{num}조"
    rest = (rest or "").strip()

    if title_in_paren:
        article_title = title_in_paren.strip()
        body_start = rest if rest else None
    else:
        # 괄호 제목이 없으면 rest 전체를 제목으로 보고, 본문은 없음
        article_title = rest if rest else None
        body_start = None

    return article_number, article_title, body_start

In [37]:
def parse_law_pdf(pdf_path: str, law_name: str) -> pd.DataFrame:
    raw_text = extract_text_from_pdf(pdf_path)
    lines = normalize_lines(raw_text, law_name=law_name)

    records = []
    current_chapter = None
    current_article_number = None
    current_article_title = None
    current_clause_number = None

    clause_buffer = []
    article_head_buffer = []

    def flush_clause():
        nonlocal clause_buffer, current_clause_number
        if clause_buffer:
            records.append({
                "law_name": law_name,
                "chapter": current_chapter,
                "article_number": current_article_number,
                "article_title": current_article_title,
                "clause_number": current_clause_number,
                "text": " ".join(clause_buffer).strip()
            })
        clause_buffer = []

    def flush_article_head():
        nonlocal article_head_buffer
        if article_head_buffer:
            records.append({
                "law_name": law_name,
                "chapter": current_chapter,
                "article_number": current_article_number,
                "article_title": current_article_title,
                "clause_number": None,
                "text": " ".join(article_head_buffer).strip()
            })
        article_head_buffer = []

    prev_article_no = None   # 👈 조 번호 추적용

    for line in lines:
        # ---------------- 장(章) ----------------
        if CHAPTER_PATTERN.match(line):
            flush_clause()
            flush_article_head()
            current_chapter = line.strip()
            continue

        # ---------------- 조(條) + 추가 보정 ----------------
        m = ARTICLE_PATTERN.match(line)
        is_article_line = False

        if m:
            # 일단 숫자 뽑아 보기
            try:
                num_now = int(re.search(r'\d+', line).group())
            except Exception:
                num_now = None

            # ⚠ 추가 보정 부분: 이전 조와 번호 차이가 너무 크면
            #    (예: 31 다음 줄에 "제21조제2항…" 같은 본문) → 조 제목이 아니라고 판단
            if num_now is not None and prev_article_no is not None \
               and abs(num_now - prev_article_no) > 5:
                # 조 제목으로 쓰지 않고, 그냥 일반 본문으로 처리하게 둠
                is_article_line = False
            else:
                is_article_line = True

        if is_article_line:
            flush_clause()
            flush_article_head()

            article_number, article_title, body_start = parse_article_title(line)
            current_article_number = article_number
            current_article_title = article_title
            current_clause_number = None

            # prev_article_no 업데이트
            try:
                prev_article_no = int(re.search(r'\d+', article_number).group())
            except Exception:
                pass

            # 같은 줄에 본문이 붙어 있는 경우 처리 (① … 또는 일반 문장)
            if body_start:
                clause_no, rest = split_clause_start(body_start)
                if clause_no:
                    current_clause_number = clause_no
                    clause_buffer = [rest] if rest else []
                else:
                    article_head_buffer.append(body_start)
            continue  # 👈 조 처리 끝났으니 다음 줄로

        # ---------------- 항(①, ②, …) ----------------
        clause_no, rest_line = split_clause_start(line)
        if clause_no:
            flush_clause()
            current_clause_number = clause_no
            clause_buffer = [rest_line] if rest_line else []
            continue

        # ---------------- 일반 본문 ----------------
        if current_article_number is None:
            continue

        if current_clause_number:
            clause_buffer.append(line)
        else:
            article_head_buffer.append(line)

    # 루프 끝나고 남은 것 flush
    flush_clause()
    flush_article_head()

    return pd.DataFrame(records, columns=[
        "law_name", "chapter", "article_number",
        "article_title", "clause_number", "text"
    ])


In [38]:
def process_all_pdfs(input_dir: str, 
                     output_dir: str,
                     merge_to_one: bool = False,
                     merged_filename: str = "ALL_LAWS.csv"):
    """
    input_dir 아래 모든 PDF(.pdf/.PDF/...)를 찾아
    - 각 PDF → <파일명>.csv
    - (옵션) 전부 합친 merged CSV 생성
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("📂 입력 폴더:", input_dir)
    print("📍 입력 절대 경로:", input_dir.resolve())
    print("📂 출력 폴더:", output_dir)
    print("📍 출력 절대 경로:", output_dir.resolve())

    # 하위 폴더까지 전부 뒤져서 .pdf 확장자 찾기
    pdf_files = [
        p for p in input_dir.rglob("*")
        if p.is_file() and p.suffix.lower() == ".pdf"
    ]

    print(f"\n🔎 찾은 PDF 개수: {len(pdf_files)}")
    if not pdf_files:
        print("⚠️ PDF를 하나도 못 찾았습니다.")
        return

    all_dfs = []

    for pdf_path in tqdm(pdf_files, desc="Processing PDF → CSV"):
        law_name = pdf_path.stem  # 파일명 그대로 사용
        try:
            df = parse_law_pdf(str(pdf_path), law_name)
        except Exception as e:
            print(f"[ERROR] {pdf_path.name} 처리 중 오류: {e}")
            continue

        # CSV 파일명도 PDF 파일명 그대로
        output_csv = output_dir / f"{pdf_path.stem}.csv"
        df.to_csv(output_csv, index=False, encoding="utf-8-sig")
        all_dfs.append(df)

    if merge_to_one and all_dfs:
        merged_df = pd.concat(all_dfs, ignore_index=True)
        merged_path = output_dir / merged_filename
        merged_df.to_csv(merged_path, index=False, encoding="utf-8-sig")
        print("\n✅ 병합 CSV 저장:", merged_path)

In [40]:
PDF_DIR = r"C:/work/03. 제안서/2025/AGI/데이터/law_pdf_downloads"  # PDF들이 있는 폴더
OUTPUT_DIR = r"C:/work/03. 제안서/2025/AGI/데이터/law_csv"         # CSV 저장할 폴더

process_all_pdfs(
    input_dir=PDF_DIR,
    output_dir=OUTPUT_DIR,
    merge_to_one=True,
    merged_filename="ALL_163_LAWS.csv",
)

📂 입력 폴더: C:\work\03. 제안서\2025\AGI\데이터\law_pdf_downloads
📍 입력 절대 경로: C:\work\03. 제안서\2025\AGI\데이터\law_pdf_downloads
📂 출력 폴더: C:\work\03. 제안서\2025\AGI\데이터\law_csv
📍 출력 절대 경로: C:\work\03. 제안서\2025\AGI\데이터\law_csv

🔎 찾은 PDF 개수: 164


Processing PDF → CSV: 100%|██████████████████████████████████████████████████████████| 164/164 [11:34<00:00,  4.24s/it]



✅ 병합 CSV 저장: C:\work\03. 제안서\2025\AGI\데이터\law_csv\ALL_163_LAWS.csv


In [14]:
from pathlib import Path

PDF_DIR = "law_pdf_downloads"  # 네가 지금 넣은 값 그대로

base = Path(PDF_DIR)
print("📂 입력한 경로:", PDF_DIR)
print("📍 절대 경로:", base.resolve())
print("존재 여부:", base.exists())
print("디렉토리인지:", base.is_dir())

print("\n폴더 안 파일 목록:")
for p in base.glob("*"):
    print(" -", p.name)


📂 입력한 경로: law_pdf_downloads
📍 절대 경로: C:\work\03. 제안서\2025\AGI\데이터\law_pdf_downloads
존재 여부: True
디렉토리인지: True

폴더 안 파일 목록:
 - .ipynb_checkpoints
 - 간선급행버스체계의 건설 및 운영에 관한 특별법.pdf
 - 감염병의 예방 및 관리에 관한 법률.pdf
 - 개인정보 보호법 시행령.pdf
 - 개인정보 보호법.pdf
 - 건강기능식품에 관한 법률.pdf
 - 건축법 시행령(대통령령)(제35811호)(20251001).pdf
 - 건축법(법률)(제21065호)(20251001).pdf
 - 게임산업진흥에 관한 법률.pdf
 - 고압가스 안전관리법 시행규칙.pdf
 - 고압가스 안전관리법 시행령.pdf
 - 고압가스 안전관리법.pdf
 - 공중위생관리법 시행령.pdf
 - 공중위생관리법.pdf
 - 공증인법.pdf
 - 공항시설법.pdf
 - 관광진흥법 시행규칙.pdf
 - 관광진흥법 시행령.pdf
 - 관광진흥법.pdf
 - 교육기본법.pdf
 - 교육환경 보호에 관한 법률.pdf
 - 국민 평생 직업능력 개발법.pdf
 - 국민연금법 시행령.pdf
 - 국민연금법.pdf
 - 국토의 계획 및 이용에 관한 법률 시행규칙.pdf
 - 국토의 계획 및 이용에 관한 법률 시행령.pdf
 - 국토의 계획 및 이용에 관한 법률.pdf
 - 근로기준법.pdf
 - 근로자퇴직급여 보장법.pdf
 - 금융산업의 구조개선에 관한 법률.pdf
 - 금융소비자 보호에 관한 법률.pdf
 - 금융실명거래 및 비밀보장에 관한 법률 시행령.pdf
 - 금융실명거래 및 비밀보장에 관한 법률.pdf
 - 금융위원회의 설치 등에 관한 법률.pdf
 - 금융지주회사법.pdf
 - 금융혁신지원 특별법.pdf
 - 기술보증기금법.pdf
 - 기초연구진흥 및 기술개발지원에 관한 법률.pdf
 - 농수산물 유통 및 가격안정에 관한 법률.pdf
 - 농어촌정비법.pdf
 - 농업기계화 촉진법.

In [12]:
import os
os.getcwd()


'C:\\work\\03. 제안서\\2025\\AGI\\데이터'

In [17]:
from pathlib import Path

OUTPUT_DIR = "C:\work\03. 제안서\2025\AGI\데이터\law_csv"  # 또는 너가 함수 호출할 때 쓴 값 그대로 넣기

out_base = Path(OUTPUT_DIR)

print("📂 OUTPUT_DIR 설정값:", OUTPUT_DIR)
print("📍 절대 경로:", out_base.resolve())
print("존재 여부:", out_base.exists(), "| 디렉토리:", out_base.is_dir())

print("\n🔎 그 안에 있는 파일들:")
for p in out_base.glob("*"):
    print(" -", p.name)


📂 OUTPUT_DIR 설정값: C:\work. 제안서5\AGI\데이터\law_csv
📍 절대 경로: C:\work. 제안서5\AGI\데이터\law_csv
존재 여부: False | 디렉토리: False

🔎 그 안에 있는 파일들:
